## 1 — Imports & Setup

Classical ML — Hyperparameter Tuning & Calibration

This notebook performs **hyperparameter tuning**, **probability calibration**, and **overfitting analysis**
for the classical ML models trained in `classical_training.ipynb`.

**Methods used:**
- GridSearchCV (exhaustive)
- RandomizedSearchCV (stochastic)
- Optuna Bayesian optimisation (TPE sampler)
- Probability calibration (CalibratedClassifierCV)
- Learning curves for overfitting detection

All tuning utilities are defined in `hyperparameter_tuning.py`.  
Structured logging is provided by `training_logger.py`.

## 1 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
from sklearn.model_selection import train_test_split, StratifiedKFold
from classical_models import (
    get_all_models, evaluate_model,
    FEATURE_COLUMNS, TARGET_RAIN, LOCATION_COLUMN, DATA_PATH,
)
from hyperparameter_tuning import (
    get_param_grids, get_param_distributions,
    run_grid_search, run_random_search,
    run_optuna_study,
    calibrate_model, compare_calibration,
    plot_learning_curves, plot_calibration_curves,
)
from training_logger import (
    setup_logger, log_training_metrics, log_tuning_results,
    log_calibration_results, log_model_save,
)

MODELS_DIR = "saved_models/"
os.makedirs(MODELS_DIR, exist_ok=True)

# Setup structured logger
logger = setup_logger("classical_finetuning")
logger.info("=== Fine-Tuning Session Started ===")

print("Imports OK")

## 2 — Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLUMNS]
y = df[TARGET_RAIN]

# Same split as training notebook
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Rain prevalence — Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")

logger.info(f"Data loaded for fine-tuning: train={X_train.shape}, test={X_test.shape}")

## 3 — Model Instantiation

In [ ]:
models = get_all_models()
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipeline in models.items():
    clf_name = type(pipeline.steps[-1][1]).__name__
    print(f"✅ {name}: {clf_name}")

## 4 — Overfitting Monitoring: Learning Curves

Learning curves plot training vs validation scores for increasing training set sizes.  
A large gap between train and val scores indicates **overfitting**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, (name, pipeline) in zip(axes, models.items()):
    print(f"Plotting learning curves for {name}...")
    logger.info(f"Plotting learning curves for {name}")
    plot_learning_curves(pipeline, X_train, y_train, cv=cv_strategy, ax=ax)
    ax.set_title(f"{name}")

plt.tight_layout()
plt.savefig("artifacts/learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Learning curves saved.")

## 5 — GridSearchCV

In [ ]:
param_grids = get_param_grids()
grid_results = {}

for name, pipeline in models.items():
    print(f"\n{'='*60}")
    print(f"GridSearch for {name}...")
    logger.info(f"Starting GridSearch for {name}")
    
    gs = run_grid_search(pipeline, param_grids[name], X_train, y_train, cv=cv_strategy)
    grid_results[name] = gs
    
    print(f"  Best ROC-AUC: {gs.best_score_:.4f}")
    print(f"  Best params: {gs.best_params_}")
    
    log_tuning_results(logger, name, "grid_search", gs.best_params_, gs.best_score_)

print("\n✅ GridSearchCV complete for all models.")

## 6 — RandomizedSearchCV

In [ ]:
param_distributions = get_param_distributions()
random_results = {}

for name, pipeline in models.items():
    print(f"\n{'='*60}")
    print(f"RandomSearch for {name}...")
    logger.info(f"Starting RandomSearch for {name}")
    
    # Re-instantiate unfitted model
    fresh_models = get_all_models()
    rs = run_random_search(
        fresh_models[name], param_distributions[name],
        X_train, y_train, n_iter=50, cv=cv_strategy
    )
    random_results[name] = rs
    
    print(f"  Best ROC-AUC: {rs.best_score_:.4f}")
    print(f"  Best params: {rs.best_params_}")
    
    log_tuning_results(logger, name, "random_search", rs.best_params_, rs.best_score_)

print("\n✅ RandomizedSearchCV complete for all models.")

## 7 — Optuna Bayesian Optimisation

In [ ]:
optuna_results = {}

for name in models.keys():
    print(f"\n{'='*60}")
    print(f"Optuna study for {name}...")
    logger.info(f"Starting Optuna study for {name}")
    
    study = run_optuna_study(name, X_train, y_train, n_trials=50, cv=cv_strategy)
    optuna_results[name] = study
    
    print(f"  Best ROC-AUC: {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")
    
    log_tuning_results(logger, name, "optuna", study.best_params, study.best_value)

print("\n✅ Optuna optimisation complete for all models.")

## 8 — Best Model Selection & Comparison

In [ ]:
# Compare best scores across all tuning methods
comparison_rows = []
for name in models.keys():
    comparison_rows.append({
        "Model": name,
        "GridSearch ROC-AUC": grid_results[name].best_score_,
        "RandomSearch ROC-AUC": random_results[name].best_score_,
        "Optuna ROC-AUC": optuna_results[name].best_value,
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
print(comparison_df.to_markdown())

# Determine best method per model
for name in models.keys():
    best_method = comparison_df.loc[name].idxmax()
    best_score = comparison_df.loc[name].max()
    print(f"\n{name}: Best method = {best_method} ({best_score:.4f})")
    logger.info(f"Best tuning for {name}: {best_method} ({best_score:.4f})")

## 9 — Probability Calibration

Use `CalibratedClassifierCV` (isotonic) to improve probability estimates.
Compare raw vs calibrated models on ROC-AUC, Brier score, and log-loss.

In [ ]:
# Use the best GridSearch models for calibration
calibrated_models = {}

for name in models.keys():
    print(f"\nCalibrating {name}...")
    logger.info(f"Calibrating {name}")
    
    best_model = grid_results[name].best_estimator_
    cal_model = calibrate_model(best_model, X_test, y_test)
    calibrated_models[name] = cal_model
    
    # Compare raw vs calibrated
    raw_vs_cal = compare_calibration(best_model, cal_model, X_test, y_test)
    
    print(f"  Raw:        ROC-AUC={raw_vs_cal['raw']['roc_auc']:.4f}, "
          f"Brier={raw_vs_cal['raw']['brier_score']:.4f}")
    print(f"  Calibrated: ROC-AUC={raw_vs_cal['calibrated']['roc_auc']:.4f}, "
          f"Brier={raw_vs_cal['calibrated']['brier_score']:.4f}")
    
    log_calibration_results(logger, name, raw_vs_cal)

print("\n✅ Calibration complete for all models.")

## 10 — Calibration Curves

In [ ]:
# Plot calibration curves for all best models
all_models_for_cal = {}
for name in models.keys():
    all_models_for_cal[f"{name} (raw)"] = grid_results[name].best_estimator_
    all_models_for_cal[f"{name} (calibrated)"] = calibrated_models[name]

fig, ax = plt.subplots(figsize=(10, 8))
plot_calibration_curves(all_models_for_cal, X_test, y_test, ax=ax)
plt.tight_layout()
plt.savefig("artifacts/calibration_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Calibration curves saved.")

## 11 — Save Best Tuned Models

In [ ]:
for name in models.keys():
    best_model = grid_results[name].best_estimator_
    model_path = os.path.join(MODELS_DIR, f"{name}.joblib")
    joblib.dump(best_model, model_path)
    print(f"Saved {name} → {model_path}")
    log_model_save(logger, f"{name}_tuned", model_path)

print("\n✅ All tuned models saved!")
logger.info("=== Fine-Tuning Session Complete ===")